# Week 04 · Evaluation and error analysis · Lab

Last week's model is 1.87 times better than guessing on the future. That number is a summary, and every summary hides something. This week we look at what it hides.

**How this lab works**

- **You write most of the code.** Cells marked **TODO** are yours, most with hints.
- Cells marked **Check** test your work, and tell you where to look if something is off.
- Several questions ask you to explain a result in your own words. Those answers are the point of the week.
- It is longer than one session **on purpose**. Parts 6 to 8 make good homework.
- You may use an AI assistant for code. You may not skip understanding what it wrote.
- Work in your own copy: *File → Save As* `my-lab.ipynb`.

## Setup

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

from sklearn.metrics import average_precision_score, precision_score, recall_score

sys.path.insert(0, "..")
from checkpoints import week_03, week_03_pipeline

RANDOM_STATE = 42
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
})
ACCENT, WARM, GREY = "#3f51b5", "#e07b39", "#c9cdd6"


def lift(y_true, scores):
    return average_precision_score(y_true, scores) / np.mean(y_true)

## 0. Start with last week's model

We rebuild the Week 3 table and the chosen pipeline, fit it on the training orders, and score the validation orders. Everything in Parts 1 to 5 happens on validation. Test waits until Part 6, and we look at it once.

In [ ]:
df = week_03()
train, valid, test = (df[df["split"] == part].copy() for part in ["train", "valid", "test"])

model = week_03_pipeline().fit(train, train["is_late"])
valid["score"] = model.predict_proba(valid)[:, 1]

print(f"validation lift: {lift(valid['is_late'], valid['score']):.2f}")

## 1. From score to decision

The model does not say *late* or *on time*. It gives each order a score between 0 and 1. To act, someone has to pick a **threshold**: orders scoring above it are flagged, the rest are not.

Every threshold is a different trade. Let's see four of them.

In [ ]:
# TODO: for each threshold in [0.5, 0.6, 0.7, 0.8]:
#       - flagged = 1 where valid["score"] >= threshold, else 0
#       - record the share flagged, the precision and the recall
#       Build a DataFrame called `thresholds`, indexed by threshold.
rows = []
for threshold in [0.5, 0.6, 0.7, 0.8]:
    ...
thresholds = ...
thresholds.round(3)

In [ ]:
# Check
assert abs(thresholds.loc[0.5, "recall"] - 0.643) < 0.002, "At 0.5, recall should be about 0.643."
assert abs(thresholds.loc[0.7, "precision"] - 0.366) < 0.002, "At 0.7, precision should be about 0.366."
print("Correct.")

Raise the threshold and fewer orders are flagged. Recall falls: more late orders slip through. Precision mostly rises: the flags that remain are more often right. At 0.8 almost nothing is flagged at all.

Here is the whole trade in one picture:

In [ ]:
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame({
    "precision": [precision_score(valid["is_late"], valid["score"] >= t, zero_division=0) for t in grid],
    "recall": [recall_score(valid["is_late"], valid["score"] >= t) for t in grid],
    "flagged": [(valid["score"] >= t).mean() for t in grid],
}, index=grid)

ax = curve.plot(color=[ACCENT, WARM, GREY], lw=2.2, figsize=(8, 3.8))
ax.set_xlabel("threshold")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_title("Every threshold is a different trade")
plt.show()

There is no "correct" threshold in this picture. Choosing one needs something the data cannot give us: **what each kind of mistake costs.**

## 2. The cost of mistakes

Back to the problem card from Week 1. Its *costly mistake* line said: missing a late order is bad, because the customer finds out the hard way. A false alarm only costs an unnecessary message.

Let's put numbers on that. Say **a missed late order costs 10** (an angry customer, perhaps a bad review) and **a false alarm costs 1** (one message sent for nothing). These are business assumptions, not facts, and a real team would agree them with the people who pay for the mistakes.

The cost of a decision is then the average cost per order:

In [ ]:
MISS_COST, ALARM_COST = 10, 1

# TODO: cost_per_order(y_true, flagged) returns the average cost per order:
#       (MISS_COST * misses + ALARM_COST * false alarms) / number of orders
#       misses:       flagged == 0 and y_true == 1
#       false alarms: flagged == 1 and y_true == 0
def cost_per_order(y_true, flagged):
    y_true, flagged = np.asarray(y_true), np.asarray(flagged)
    ...

# TODO: the cost at every threshold in `grid`, as a Series indexed by threshold,
#       then the threshold with the lowest cost
costs = ...
best = ...

nobody = cost_per_order(valid["is_late"], np.zeros(len(valid)))
everybody = cost_per_order(valid["is_late"], np.ones(len(valid)))
print(f"best threshold on validation: {best:.2f}, cost {costs[best]:.3f} per order")
print(f"flag nobody: {nobody:.3f}    flag everybody: {everybody:.3f}")

In [ ]:
# Check
assert abs(best - 0.48) < 0.005, "The cheapest threshold on validation should be 0.48."
assert abs(nobody - 1.184) < 0.002, "Flagging nobody means every late order is a miss."
print("Correct.")

In [ ]:
ax = costs.plot(color=ACCENT, lw=2.2, figsize=(8, 3.6), label="the model, at each threshold")
ax.axhline(nobody, color=GREY, ls="--", lw=1.5, label="flag nobody")
ax.axvline(best, color=WARM, lw=1.5, label=f"cheapest: {best:.2f}")
ax.set_xlabel("threshold")
ax.set_ylabel("cost per order")
ax.set_title("Choosing the threshold from the cost of mistakes")
ax.legend(frameon=False)
plt.show()

At a threshold of 0.48, the model cuts the cost of mistakes from 1.18 per order to 0.67, a saving of over 40% against doing nothing. That is what the model is *for*.

One more number is worth remembering. Flagging an order is worth it only if the chance it is really late is above **the cost of a false alarm divided by the cost of a miss**: here 1 / 10, so **10%**. That is the **break-even precision**. Below it, every flag costs more than it saves.

In [ ]:
BREAK_EVEN = ALARM_COST / MISS_COST
valid["flag"] = (valid["score"] >= best).astype(int)
print(f"break-even precision: {BREAK_EVEN:.0%}")
print(f"precision at {best:.2f} on validation: {precision_score(valid['is_late'], valid['flag']):.1%}")

## 3. Are these probabilities?

It is tempting to read a score of 0.8 as "80% chance of being late". Let's test that. If the scores were probabilities, orders scored around 0.3 would be late about 30% of the time.

In [ ]:
print(f"average score: {valid['score'].mean():.3f}    actual late rate: {valid['is_late'].mean():.3f}")

# TODO: group the validation orders into score bins, and for each bin compute
#       average_score   the mean score
#       actually_late   the share that were late
#       orders          how many orders
#       Hint: bins = pd.cut(valid["score"], [0, 0.2, 0.4, 0.6, 0.8, 1.0])
#             valid.groupby(bins, observed=True).agg(average_score=("score", "mean"), ...)
reliability = ...
reliability.round(3)

In [ ]:
# Check
assert len(reliability) == 5 and reliability["orders"].sum() == len(valid), "Five bins, covering every validation order."
assert reliability["actually_late"].iloc[-1] < 0.4, "Look at the top bin: are those orders late 80% of the time?"
print("Correct.")

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.plot([0, 1], [0, 1], color=GREY, ls="--", label="if scores were probabilities")
ax.plot(reliability["average_score"], reliability["actually_late"], color=WARM, marker="o", lw=2.2, label="our model")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("average score")
ax.set_ylabel("share actually late")
ax.set_title("Scores are not probabilities")
ax.legend(frameon=False, loc="upper left")
plt.show()

**Your turn:** the model's scores put orders in a good order, but are too high. Why does that not hurt the lift or the threshold choice, but would hurt someone who reads a score as a chance?

*Your answer:* ...

## 4. The average hides the failures

At the chosen threshold, the model catches about two in three late orders. That is an average over very different orders. Let's break it down.

In [ ]:
REGION = {
    "AC": "North", "AM": "North", "AP": "North", "PA": "North", "RO": "North", "RR": "North", "TO": "North",
    "AL": "Northeast", "BA": "Northeast", "CE": "Northeast", "MA": "Northeast", "PB": "Northeast",
    "PE": "Northeast", "PI": "Northeast", "RN": "Northeast", "SE": "Northeast",
    "DF": "Center-West", "GO": "Center-West", "MT": "Center-West", "MS": "Center-West",
    "ES": "Southeast", "MG": "Southeast", "RJ": "Southeast", "SP": "Southeast",
    "PR": "South", "RS": "South", "SC": "South",
}
valid["region"] = valid["customer_state"].map(REGION)
valid["route"] = np.where(valid["same_state"] == 1, "within one state", "across states")
valid["distance"] = pd.cut(valid["distance_km"], [-1, 100, 500, 1000, 5000],
                           labels=["under 100 km", "100 to 500 km", "500 to 1000 km", "over 1000 km"])

In [ ]:
# TODO: slice_report(frame, by) returns one row per group of frame.groupby(by, observed=True),
#       with four columns: "orders", "late rate", "recall", "precision".
#       Use recall_score(group["is_late"], group["flag"], zero_division=0), and the same for precision.
#       Hint: collect a dict of dicts, then pd.DataFrame(rows).T
def slice_report(frame, by):
    ...

print(f"overall recall: {recall_score(valid['is_late'], valid['flag']):.3f}")
slice_report(valid, "route").round(3)

In [ ]:
# Check
report = slice_report(valid, "route")
assert abs(report.loc["within one state", "recall"] - 0.148) < 0.002, "Recall within one state should be about 0.148."
print("Correct.")

Overall recall is 68%. For orders that cross a state border, it is 78%. For orders that stay **within one state, it is 15%**. The model barely catches a single local late order.

Three more ways to slice:

In [ ]:
display(slice_report(valid, "region").round(3))
display(slice_report(valid, "distance").round(3))
display(slice_report(valid[valid["customer_state"].isin(["SP", "RJ"])], "customer_state").round(3))

In [ ]:
slices = pd.concat([
    slice_report(valid, "route"),
    slice_report(valid, "distance"),
    slice_report(valid[valid["customer_state"].isin(["SP", "RJ"])], "customer_state").rename(
        index={"SP": "to São Paulo", "RJ": "to Rio de Janeiro"}),
])["recall"].sort_values()

overall = recall_score(valid["is_late"], valid["flag"])
ax = slices.plot.barh(color=[WARM if r < overall else ACCENT for r in slices], figsize=(7, 3.8))
ax.axvline(overall, color="#333333", ls="--", lw=1.2)
ax.text(overall + 0.01, len(slices) - 0.6, "overall", fontsize=9)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlabel("recall: share of late orders the model catches")
ax.set_title("Where the average hides the failures")
plt.show()

**Your turn:** describe, in one sentence, which late orders the model catches and which it misses. Why would a single overall recall number hide this?

*Your answer:* ...

## 5. Read the mistakes

Slices tell you *where* the model fails. Reading the mistakes tells you *why*. First, the late orders it missed, compared with the late orders it caught.

In [ ]:
late = valid[valid["is_late"] == 1].copy()
late["days_late"] = (late["order_delivered_customer_date"].dt.normalize() - late["order_estimated_delivery_date"]).dt.days
late["outcome"] = np.where(late["flag"] == 1, "caught", "missed")

# TODO: one row for "caught" and one for "missed", with:
#       orders               how many
#       within_one_state     the share with same_state == 1
#       median_distance_km   the median distance
#       to_sao_paulo         the share going to SP.  Hint: ("customer_state", lambda s: (s == "SP").mean())
#       median_days_late     the median days_late
profile = ...
profile.round(2)

In [ ]:
# Check
assert profile.loc["missed", "orders"] == 520, "There should be 520 missed late orders."
assert abs(profile.loc["missed", "within_one_state"] - 0.42) < 0.01, "Within one state should be the mean of same_state."
print("Correct.")

Two different kinds of late order. The ones the model catches travel far, cross borders, and are very late, over a week past the promise. The ones it misses stay local, half of them go to São Paulo, and they are *less* late: a few days past the promise.

The model catches the long-haul disasters and misses the local near-misses.

Now look at individual orders: the late orders the model was most sure would be on time, and the on-time orders it was most sure would be late.

In [ ]:
show = ["customer_state", "seller_state", "distance_km", "window_days", "category", "score"]

print("Most confident misses: late, but scored lowest")
display(valid[valid["is_late"] == 1].nsmallest(10, "score")[show].round(2))

print("Most confident false alarms: on time, but scored highest")
display(valid[valid["is_late"] == 0].nlargest(10, "score")[show].round(2))

**Your turn:** write two hypotheses about what causes the misses, and what feature or data could capture each one.

*Hypothesis 1:* ...

*Hypothesis 2:* ...

## 6. The one look at test

Every choice is made: the features (Week 3), the model (Week 3), and the threshold (Part 2). Now we look at test, once, and report honestly.

A single number is not enough. The test set is one sample of the future, and a different sample would give a different score. The **bootstrap** tells us how much: resample the test orders with replacement a thousand times, and see how much the lift moves. We also compare with Week 1's rule on each resample, so we know whether the improvement is real.

In [ ]:
test["score"] = model.predict_proba(test)[:, 1]
rule_rate = train.groupby("customer_state")["is_late"].mean()
test["rule"] = test["customer_state"].map(rule_rate).fillna(train["is_late"].mean())

y, s, r = test["is_late"].to_numpy(), test["score"].to_numpy(), test["rule"].to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
model_lifts, gains = [], []

# TODO: repeat 1000 times:
#       i = rng.integers(0, len(y), len(y))              resample the test orders
#       append lift(y[i], s[i]) to model_lifts
#       append lift(y[i], s[i]) - lift(y[i], r[i]) to gains
#       Then the 2.5th and 97.5th percentiles of each list, with np.percentile.


print(f"test lift: {lift(y, s):.2f}   95% interval {low:.2f} to {high:.2f}")
print(f"gain over Week 1's rule: 95% interval {gain_low:.2f} to {gain_high:.2f}")

In [ ]:
# Check
assert len(model_lifts) == 1000, "Resample 1000 times."
assert abs(low - 1.73) < 0.03 and abs(high - 2.06) < 0.03, "The interval should be about 1.73 to 2.06."
assert gain_low > 0, "The whole interval for the gain should be above zero."
print("Correct.")

The lift is 1.87, and a fair range for it is about 1.7 to 2.1. The gain over Week 1's rule is between about 0.7 and 0.9, and never near zero. **The model is genuinely better at ranking.** Everything from Weeks 2 and 3 held up.

Now the decision. We apply the threshold chosen on validation, and count the cost.

In [ ]:
test["flag"] = (test["score"] >= best).astype(int)

decision = pd.DataFrame({
    "validation": [valid["is_late"].mean(), valid["flag"].mean(), precision_score(valid["is_late"], valid["flag"]),
                   cost_per_order(valid["is_late"], valid["flag"]), cost_per_order(valid["is_late"], np.zeros(len(valid)))],
    "test": [test["is_late"].mean(), test["flag"].mean(), precision_score(test["is_late"], test["flag"]),
             cost_per_order(test["is_late"], test["flag"]), cost_per_order(test["is_late"], np.zeros(len(test)))],
}, index=["late rate", "share flagged", "precision", "cost: the model", "cost: flag nobody"])
decision.round(3)

In [ ]:
ax = decision.loc[["cost: the model", "cost: flag nobody"]].T.plot.bar(color=[ACCENT, GREY], rot=0, figsize=(6.5, 3.6), width=0.6)
ax.set_ylabel("cost per order")
ax.set_title("The same decision, in two different months")
ax.legend(frameon=False)
plt.show()

On validation, the model's decision saves over 40%. On test, **it costs more than doing nothing at all.**

Take a moment with that. The model ranks orders just as well as before; the interval above proves it. The threshold is the one we chose carefully. And still the decision lost money. What happened?

## 7. What happened?

Look at the precision row. On validation, 22% of flagged orders were really late, comfortably above the 10% break-even. On test, only 7% were. Every flag now costs more than it saves.

The reason is the late rate. Validation covered March and April 2018, including the spike. Test covers the calm months after it. When late orders are rare, even well-ranked flags are mostly false alarms. Month by month:

In [ ]:
monthly = test.groupby(test["order_purchase_timestamp"].dt.to_period("M")).apply(
    lambda g: pd.Series({
        "late rate": g["is_late"].mean(),
        "average score": g["score"].mean(),
        "precision": precision_score(g["is_late"], g["flag"], zero_division=0),
    }),
    include_groups=False,
)
monthly.round(3)

Two things stand out. Precision is below the 10% break-even in three of the four test months, and only just above it in May. And the model's average score barely moves: between 0.40 and 0.54, whether 6% of orders are late or 1%. **The model does not know the world has changed.** It was never told.

So what would a team actually do? Three things, none of which involve a better algorithm:

- **Re-choose the threshold regularly**, on the most recent data, because the right threshold depends on how common late orders are right now
- **Watch the late rate**, and the share of flags that turn out right, so a change like this is noticed in weeks rather than months
- **Plan by capacity** rather than by a fixed threshold, if the team can only contact so many customers a day

As a diagnosis, not a choice, here is what flagging only the top 10% of orders would have done:

In [ ]:
for name, frame in [("validation", valid), ("test", test)]:
    top = (frame["score"] >= frame["score"].quantile(0.9)).astype(int)
    print(f"{name:10}  top 10%: precision {precision_score(frame['is_late'], top):.3f}, "
          f"cost {cost_per_order(frame['is_late'], top):.3f} vs flag nobody {cost_per_order(frame['is_late'], np.zeros(len(frame))):.3f}")

We found this out by looking at test once, so we cannot now use test to pick a new rule: that would make the test score dishonest.

**Your turn:** in your own words, why did a model that ranks just as well on test make a worse decision there? What would you monitor after launch to catch this early?

*Your answer:* ...

## 8. The model card

A **model card** is a short, honest document that travels with a model. It says what the model does, how it was built and evaluated, where it fails, and when it should not be used. It is the synthesis of everything in Block 1, and in your projects it lives in `MODEL_CARD.md`, next to `PROBLEM.md`.

**TODO:** fill it in, using the numbers from this notebook and the last three. Double-click to edit.

| | |
|---|---|
| **What it predicts** | ... |
| **Intended use** | ... |
| **Not for** | ... |
| **Data** | ... |
| **Split** | ... |
| **Features** | ... |
| **Model** | ... |
| **Performance** | ... |
| **Where it fails** | ... |
| **Decision** | ... |
| **Before using it** | ... |

## Block 1 in one chart

Four weeks, one question, and one number that kept changing as we learned to measure it honestly.

In [ ]:
journey = pd.DataFrame({
    "lift": [1.51, 1.18, lift(y, s)],
    "low": [np.nan, np.nan, low],
    "high": [np.nan, np.nan, high],
}, index=["Week 1\nrandom split", "Week 2\nsplit by time", "Week 3 and 4\nnew features, on test"])

fig, ax = plt.subplots(figsize=(7.5, 3.8))
colours = [GREY, GREY, ACCENT]
ax.bar(range(3), journey["lift"], color=colours, width=0.55)
ax.errorbar(2, journey["lift"].iloc[2], yerr=[[journey["lift"].iloc[2] - low], [high - journey["lift"].iloc[2]]],
            color="#333333", capsize=6, lw=1.5)
for i, value in enumerate(journey["lift"]):
    ax.text(i - 0.2, value + 0.04, f"{value:.2f}", ha="center", fontsize=10)
ax.axhline(1, color="#333333", ls="--", lw=1)
ax.set_xticks(range(3))
ax.set_xticklabels(journey.index)
ax.set_ylabel("lift over guessing")
ax.set_ylim(0, 2.4)
ax.set_title("The journey of one number")
plt.show()

## What we learned

Write, in one sentence each:

1. Why is there no single "correct" threshold without knowing the cost of mistakes?
2. What did slicing the recall reveal that the overall number hid?
3. What does the bootstrap interval tell you that the single test score does not?
4. Why did the same decision save money on validation and lose money on test?

**That closes Block 1.** Project 1 asks you to do everything from these four weeks, on data you have not seen in class.